# Helper - Convert files from iOS app to server format #

We can run this instead of setting up server and everything...

In [ ]:
import json
import sys
import shutil
from pathlib import Path
from typing import Optional, Tuple, Literal

import numpy as np
from PIL import Image

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

In [10]:
def _ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def _read_json(p: Path) -> dict:
    with open(p, "r") as f:
        return json.load(f)

def _intrinsics_columns_from_flat(flat: list[float]) -> list[list[float]]:
    import numpy as np
    M = np.array(flat, dtype=np.float32).reshape(3, 3)

    # Heuristic: where is cx? If at M[0,2], it's standard K (case 1). If at M[2,0], it's K^T (case 2).
    cx_like_row_major = float(M[0, 2])  # row-major K puts cx here
    cx_like_transposed = float(M[2, 0]) # K^T row-major (your data) puts cx here

    if abs(cx_like_row_major) > abs(cx_like_transposed):
        # Case 1: standard K in row-major -> take true columns
        return [M[:, 0].tolist(), M[:, 1].tolist(), M[:, 2].tolist()]
    else:
        # Case 2: K^T in row-major (your current export) -> rows already equal desired columns
        return [M[0, :].tolist(), M[1, :].tolist(), M[2, :].tolist()]

def _decode_depth_png_to_float32(
    depth_png_path: Path,
) -> np.ndarray:
    """
    Convert depth.png to meters (float32).
    - u16_mm:       uint16 image where value is millimeters (meters = val / 1000). 'scale' ignored (uses 1000).
    """
    img = Image.open(depth_png_path)
    # Convert to 16-bit if not already
    if img.mode != "I;16":
        # Some pipelines save as L but with 16-bit file; enforce conversion
        img = img.convert("I;16")
    arr = np.array(img, dtype=np.uint16)
    depth_m = (arr.astype(np.float32)) / 1000.0
    return depth_m.astype(np.float32)

def _write_float_bin(path: Path, arr: np.ndarray) -> None:
    arr.astype(np.float32).tofile(path)

def _write_u8_bin(path: Path, arr: np.ndarray) -> None:
    arr.astype(np.uint8).tofile(path)

In [11]:
def convert_vignette_dir(
    input_dir: Path,
    output_dir: Path,
    *,
    subject_uv: Optional[Tuple[float, float]] = None,
    # Allow custom file names if your iPhone export uses different names:
    rgb_name: str = "rgb.png",
    depth_name: str = "depth.png",
    confidence_name: Optional[str] = "confidence.png",
    vignette_json_name: str = "vignette.json",
) -> None:
    """
    Convert a single vignette folder exported from the iPhone app into the server's
    expected format: rgb.png, depth.bin, (optional) confidence.bin, metadata.json.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    _ensure_dir(output_dir)

    # 1) Load vignette.json (produced by SpatialVignette.toVignetteData)
    meta_src = _read_json(input_dir / vignette_json_name)

    # 2) Copy/prepare RGB
    rgb_src = input_dir / (meta_src.get("paths", {}).get("rgb") or rgb_name)
    rgb_dst = output_dir / "rgb.png"
    if not rgb_src.exists():
        # fallback to default name
        rgb_src = input_dir / rgb_name
    if not rgb_src.exists():
        raise FileNotFoundError(f"RGB image not found at {rgb_src}")
    shutil.copy2(rgb_src, rgb_dst)

    # 3) Convert depth.png -> depth.bin (float32 meters)
    depth_src = input_dir / (meta_src.get("paths", {}).get("depth") or depth_name)
    if not depth_src.exists():
        depth_src = input_dir / depth_name
    if not depth_src.exists():
        raise FileNotFoundError(f"Depth image not found at {depth_src}")
    depth_m = _decode_depth_png_to_float32(depth_src)
    _write_float_bin(output_dir / "depth.bin", depth_m)

    # 4) Convert confidence.png -> confidence.bin (uint8), if present
    wrote_confidence = False
    conf_path_in_meta = (meta_src.get("paths", {}) or {}).get("confidence")
    if conf_path_in_meta:
        conf_src = input_dir / conf_path_in_meta
        if conf_src.exists():
            img_conf = Image.open(conf_src).convert("L")
            conf_arr = np.array(img_conf, dtype=np.uint8)
            _write_u8_bin(output_dir / "confidence.bin", conf_arr)
            wrote_confidence = True
    if not wrote_confidence and confidence_name:
        # fallback to default name
        conf_src = input_dir / confidence_name
        if conf_src.exists():
            img_conf = Image.open(conf_src).convert("L")
            conf_arr = np.array(img_conf, dtype=np.uint8)
            _write_u8_bin(output_dir / "confidence.bin", conf_arr)
            wrote_confidence = True

    # 5) Build server-side metadata.json
    cam = meta_src.get("camera", {})
    res = cam.get("resolution", {})
    width = int(res.get("width"))
    height = int(res.get("height"))
    flat_intr = cam.get("intrinsics")
    if not isinstance(flat_intr, list) or len(flat_intr) != 9:
        raise ValueError("camera.intrinsics in vignette.json must be a 9-element row-major list")

    intr_columns = _intrinsics_columns_from_flat(flat_intr)

    # Subject UV
    if subject_uv is None:
        subject_uv = (0.5, 0.5)

    metadata_out = {
        "resolution": [width, height],                          # [w, h]
        "camera_intrinsics": {"columns": intr_columns},         # columns-format
        "subject_uv": [float(subject_uv[0]), float(subject_uv[1])]
    }

    with open(output_dir / "metadata.json", "w") as f:
        json.dump(metadata_out, f, indent=2)

    print(f"✓ Wrote: {rgb_dst.name}, depth.bin"
          f"{', confidence.bin' if wrote_confidence else ''}, metadata.json to {output_dir}")

In [ ]:
name = "branches"

IN_PATH = project_root / "data" / "Vignettes" / name
OUT_PATH = project_root / "test_data" / name

convert_vignette_dir(
    IN_PATH,
    OUT_PATH,
    subject_uv=None
)

✓ Wrote: rgb.png, depth.bin, confidence.bin, metadata.json to /Users/yuzhenzhang/Documents/Research/TestApp/SpatialVignetteServer/test_data/book
